# Pitch benchmark

This notebook compares Attune's production pYIN+ method against monophonic pitch trackers over the same corpora and `mir_eval.melody` scoring.

The primary rows have distinct, defensible meanings:

- **`pyin`** — Attune's local, algorithmically equivalent reimplementation of `librosa.pyin`.
- **`attune`** — the shipped ordinary-candidate pYIN+ pipeline. Real time applies the causal raw-confidence threshold `0.985` plus a `0.03 × running-peak RMS` floor. Completed audio now uses pitch-only confidence-dependent emissions with 50 ms gap bridging, followed by a framewise `0.985` confidence threshold and `0.04 × p95(RMS)` floor. Decoded bins without positive candidate support are marked unvoiced after tracking; supported pitches and the HMM path are unchanged.
The `attune` row calls the production classes in `algorithms/` through a thin benchmark adapter. The local `pyin` row has its own versioned cache, while other competitors keep their per-track estimate caches so interrupted runs resume.

Saved suite rows carry the production pitch-cache version. When that version changes, the suite runner automatically refreshes only `attune` and merges it into the cached third-party rows; ordinary notebook reruns do not force any detector. Offline scoring changes reuse valid per-track estimates; incompatible algorithm caches or missing tracks still require computation. The new original-Bach10 suite runs every configured available method and resumes its own caches. `PRIMARY_METHODS` and the cache identity are notebook-level constants below so experiment membership stays visible.

**Basic Pitch interpretation:** `basic_pitch` selects the strongest in-range bin from the 264-bin contour head independently at every frame, while the note head supplies only its standard 0.3 voicing gate.

Reported `Audio(s)/Compute(s)` is measured on the first uncached pass. Suites cache rows under `results/pitch/notebook_runs/<suite>/`. Ordinary reruns reuse completed rows. Score-only refreshes read valid detector caches; missing or incompatible algorithm caches require computation. Older streaming checkpoints without the frame counts needed for weighted scoring require a one-time replay. Volume, threshold, and voicing parameter sweeps live in `benchmarks/notebooks/sweeps/pitch.ipynb`.

**Run All:** Compare all configured available methods on the original, isolated Bach10 recordings. URMP, CocoChorales, degraded-audio, and streaming cells keep their existing Attune-only refresh policy. Full-corpus suites remain opt-in with `RUN_FULL_RUNS = True`; the full audio suite also runs all configured methods. The leaderboard uses preliminary suites by default.
All five pitch metrics pool their own frame numerators and denominators on a common 10 ms grid. Long recordings contribute proportionally more; no headline metric averages track or dataset percentages. Old score rows are refreshed from estimate caches when possible. Streaming checkpoints without frame counts require replay.


### Fair competitor configuration

Every external estimate is now given the same reference/score-derived admissible F0 range. APIs that support range-aware candidate selection (for example Attune, Praat, and Basic Pitch) use it internally; fixed-output APIs are also range-gated after inference. This distinction matters for SwiftF0: its [official API](https://github.com/lars76/swift-f0/) says the network always detects over 46.875–2093.75 Hz and `fmin`/`fmax` only decide which outputs count as voiced. The shared final gate therefore supplies the same information without claiming that it changes SwiftF0's CNN inference. Praat remains raw AC because Praat documents it as the periodicity method; its filtered recommendation is aimed at speech intonation and vocal-fold analysis.

SPICE uses one fixed confidence threshold of **0.35** across every dataset and the streaming adapter. Predictions below it remain unvoiced for scoring. The threshold was chosen after the exploratory confidence sweep; these preliminary results are not a held-out estimate of threshold selection performance.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import sys

REPO_ROOT = next(
    (
        path
        for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (path / "app.py").is_file()
    )
)
sys.path.insert(0, str(REPO_ROOT))
import importlib

for module_name in (
    "algorithms.PitchSmoother",
    "benchmarks.modules.pitch.PitchCache",
    "benchmarks.modules.pitch.competitors.Attune",
    "benchmarks.modules.pitch.competitors.Spice",
    "benchmarks.modules.pitch.PitchBenchmarker",
    "benchmarks.modules.pitch.PitchBenchmarker",
    "benchmarks.modules.pitch.datasets.Bach10",
    "benchmarks.modules.pitch.PitchBenchmarker",
    "benchmarks.modules.pitch.PitchNotebook",
):
    importlib.reload(importlib.import_module(module_name))
from benchmarks.modules.pitch.PitchNotebook import PitchNotebook


In [3]:
PRIMARY_METHODS = ("pyin", "attune", "praat", "spice")
RUN_FULL_RUNS = False
CONFIG = PitchNotebook.NotebookConfig(use_cache=True, force_rerun=False)
BENCH = PitchNotebook(CONFIG)
CACHE_METHODS = tuple(BENCH.methods)
BENCH.show_configuration()
RUN_PAPER_EVALUATION = True
OFFLINE_METHODS = tuple(BENCH.methods)
ONLINE_METHODS = ("attune", "pyin_framewise", "praat", "swiftf0", "spice")
PAIRED_RESAMPLES = 9999


,setting,value
0,workers,8
1,streaming workers,8
2,preliminary tracks per instrument,2
3,full CocoChorales stems per stratum,all
4,random seed,0
5,methods,"pyin, attune, basic_pitch, praat, swiftf0, cre..."
6,estimate caches,reused
7,degradation SNRs,"clean, 20dB, 15dB, 10dB, 5dB"
8,run cache,/Users/sarah/Desktop/shasha/Attune/benchmarks/...


## Method availability

Third-party trackers are optional dependencies, and a missing one is skipped rather than fatal. Run this first so a long cell later does not quietly drop a method: anything marked `MISSING` carries the install hint that would make it available. `torchcrepe` and `penn` are excluded by default because they stack a second heavy PyTorch runtime; set `include_slow_methods=True` on the config to include them, ideally in their own invocation.

In [ ]:
availability = BENCH.show_availability()


## Preliminary sampling check

Every preliminary suite uses the same sampling contract: `CONFIG.preliminary_tracks_per_instrument` tracks for every instrument label, selected independently within each label using `CONFIG.seed`. With the defaults this yields 8 original Bach10 isolated recordings (4 instruments × 2), 26 URMP stems (13 × 2), and 26 CocoChorales stems (13 × 2). The check below also confirms that the same seed repeats exactly while the next seed changes track identities without changing those quotas.

In [ ]:
preliminary_sampling = BENCH.show_preliminary_sampling()


## Preliminary comparison — original Bach10 recordings

Two seed-selected recordings from each of Bach10's four instruments (violin, clarinet, saxophone, bassoon; 8 total). Only isolated monophonic stems are evaluated; ensemble mixes are excluded. Each method's search range is tightened to the reference melody's range.

These are real performances, replacing `bach10-mf0-synth` in this suite. The supplied `*-GTF0s.mat` annotations contain fractional MIDI pitches: the loader converts them to Hz, preserves unvoiced frames, and uses the documented 10 ms hop with the first frame centered at 23 ms. These references were produced with YIN and manual corrections, so they are not independent physical measurements of F0. See the [official description](https://labsites.rochester.edu/air/datasets/Bach10%20Dataset_v1.0.pdf).

Data location: `benchmarks/datasets/bach10-original/<piece>/`. The local v1.1 release comes from this [public mirror, pinned revision](https://github.com/flippy-fyp/Bach10_v1.1/tree/2a53cdc6495bb82f03f943b77ca3b11ddf7f5a31). To provision another checkout, extract that archive into the data location, retaining the ten piece directories. Missing original data raises an error; synthesized audio is never substituted.

This cell runs all configured available methods, then resumes cached results on subsequent runs. Original results use `preliminary_bach10_original`; historical synthesized results under `preliminary` remain unchanged.


In [ ]:
preliminary = BENCH.run_preliminary(cache_methods=CACHE_METHODS)


## Preliminary URMP comparison

Run the same detector set on two seed-selected isolated stems from each of URMP's 13 instrument classes (26 total). The adapter pairs each `AuSep` WAV with its 10 ms `F0s` annotation and carries the piece, ensemble size, instrument, and part number into the result rows. This is the quickest balanced check that the downloaded corpus and every installed method work end to end.

In [ ]:
preliminary_urmp = BENCH.run_preliminary_urmp(force=False, cache_methods=CACHE_METHODS)


### Low-latency real-time replay

This live comparison has one controlled pair, one explicitly frame-restarted local pYIN diagnostic, and two neural adapters. Attune's ordinary-candidate frame-local pYIN with the selected causal `0.985` confidence and `0.03 × running-peak RMS` gates and raw Praat receive the identical 44.1-kHz capture frame, hop, center timestamp, and score-derived nominal pitch range. `pyin_framewise` runs the local implementation on each newly available 4096-sample, left-aligned frame (`center=False`) at 44.1 kHz. It uses the reference single-frame joint voiced/unvoiced state decision, without an extra 0.5 probability gate or temporal smoothing; it remains distinct from the full-track `pyin` baseline.

SwiftF0 and SPICE use their smallest practical model-native rolling center frames; every update commits one estimate only after its required future samples arrive. SwiftF0 uses 320 ms of receptive context / about 152 ms look-ahead at 16 ms updates, SPICE 64 ms / 32 ms at 32 ms updates. RMVPE is excluded from online runs because its pilot configuration accumulated backlog; it remains in the offline comparison. There is no whole-file call, 1--2 second bounded-history rerun, or timestamp ablation in this table. Model loading and one warm-up call happen before steady-state timing.

The first table uses the notebook's standard pitch-summary columns. The second retains per-recording interaction diagnostics with the same title-case naming convention, and the third shows each adapter's native geometry while making the Attune/Praat match explicit and the frame-restarted librosa geometry visible. Attune uses a `0.03` running-peak RMS floor in live mode and a `0.04` whole-track p95 RMS floor post hoc. Pass `instruments=()` to sample one stem from every selected corpus instrument, and set `force=True` only when intentionally replacing otherwise current rows.

**Latency retention fix (2026-10-04):** v7 saved only per-recording percentiles, so its exact pooled P95 cannot be recovered. v9 saves every output-update latency, CPU duration, and estimate timestamp in compressed checkpoint sidecars. Ordinary reruns now use a new versioned folder and rerun all methods once; completed v9 recordings are reused only while their latency samples remain valid. Existing v7 results are preserved. Restart the kernel before rerunning after this code update.

Methods use up to all but two logical CPUs (`CONFIG.streaming_workers` sets the ceiling), with concurrency reduced according to available RAM before each method, reusing warmed models within each worker. Progress shows the latest completed track with an elapsed timer. Each completed song/method is checkpointed immediately; rerun with `force=False` to resume, or `force=True` to recompute. Frames remain sequential. Like `pyin_ablation.ipynb`, output latency and deadline misses use dedicated-worker process CPU simulation; measured wall compute time remains a separate diagnostic and can increase under contention. The timing cache version separates these results from the older wall-latency rows.

**pYIN correction (2026-10-04):** v9 removes the nonstandard 50% voiced-probability cutoff. The single-frame decision compares the strongest voiced state with one unvoiced state, matching reference one-frame Viterbi with uniform initialization. Restart the kernel and rerun the online cells; `force=False` uses fresh v9 output folders and preserves older results. Previous streaming pYIN accuracy and paired comparisons must be replaced.


The scheduler reserves the larger of 2 GiB or 15% of physical RAM for the desktop, then budgets 1 GiB per classical streaming worker or 2 GiB per neural/offline worker. These are estimates, not hard memory limits. RAM is rechecked between methods/batches. Each streaming method releases its worker processes when it finishes. Closing unused applications can allow more workers on the next method/batch.


In [ ]:
streaming_urmp = BENCH.run_streaming_comparison(
    instruments=(), tracks_per_instrument=2, force=False
)


## Preliminary CocoChorales comparison

The same methods over two seed-selected stems from each of CocoChorales' 13 instrument classes (26 total). The helper prefers two different source compositions within each class and materializes the selected stems automatically. Rows carry ensemble/instrument metadata, and each search range is taken from a robust percentile of the annotated f0 rather than its extremes.

In [ ]:
preliminary_coco = BENCH.run_preliminary_coco(force=False, cache_methods=CACHE_METHODS)


## Offline paired analysis

Pool all five metrics across the eight Bach10, 26 URMP, and 26 CocoChorales preliminary tracks. Compare Attune with each offline competitor on pooled overall accuracy using paired source-piece label swaps and 95% paired cluster-bootstrap intervals. Apply Holm correction within the offline family. Every comparison and the headline table use the same completed track intersection; coverage and excluded tracks are reported. Missing methods raise an error rather than silently changing the test family.

Frames retain equal scoring weight, while dependent frames and stems stay together for inference. `source_groups` can link known shared compositions across corpora. Intervals are unadjusted; adjusted p-values determine significance. These development samples do not establish held-out generalization.


In [ ]:
offline_paired = BENCH.show_paired_analysis(
    preliminary,
    preliminary_urmp,
    preliminary_coco,
    mode="Offline",
    methods=OFFLINE_METHODS,
    n_resamples=PAIRED_RESAMPLES,
)


## Online paired analysis

Add the same eight Bach10 and 26 CocoChorales tracks to the 26-track URMP streaming replay. Pool all five metrics across datasets, then compare overall accuracy, voicing recall, and voicing false-alarm rate against each of the five online competitors. Holm correction covers these fifteen online tests separately from offline tests. Source-piece pairing, confidence intervals, coverage checks, and frame weights follow the offline analysis. The streaming pYIN adaptation is included in all three paired comparisons.


In [ ]:
streaming_bach10 = BENCH.run_streaming_comparison(
    dataset="bach10-original", instruments=(), tracks_per_instrument=2
)
streaming_coco = BENCH.run_streaming_comparison(
    dataset="coco", instruments=(), tracks_per_instrument=2
)
online_paired = BENCH.show_paired_analysis(
    streaming_bach10,
    streaming_urmp,
    streaming_coco,
    mode="Streaming",
    methods=ONLINE_METHODS,
    n_resamples=PAIRED_RESAMPLES,
)


## Recommended expanded evaluation

Analyze **449 tracks in each mode: 40 Bach10, 149 URMP, and 260 CocoChorales (20 per instrument)**, using the same selection for offline and online runs. This covers all real recordings and a larger instrument-balanced synthetic sample. It is a starting target, not a promise of significance: power depends on the effect size and independent source pieces. Bach10 and URMP contain 10 and 44 pieces, respectively.

Before the final runs, freeze configurations and use pilot piece-level counts to check power for meaningful differences (suggested planning targets: 0.5 percentage points in overall accuracy and 1 point in each online voicing metric, at 90% power with the same correction). Increase independent-piece coverage if needed; do not keep sampling until a p-value becomes significant. Identify previously used development pieces and report genuinely held-out evidence separately. Inspect shared-composition IDs before inference.

The following opt-in cells run the clean selection and write aggregate metrics, dataset-specific appendix tables, coverage, paired rows, and statistical results.


The final **449-track streaming run must use v9 latency-sample checkpoints**. The versioned run paths enforce a fresh replay instead of reusing summary-only v7 results, even with `force_rerun=False`. The cell below exports exact pooled P95 by concatenating individual output-update latencies per method across recordings (each output update has equal weight). Keep the `.latency.npz` sidecars with the results; per-recording P95 averages are not pooled P95. The expanded run is enabled by `RUN_PAPER_EVALUATION` in the configuration cell. Set it to `False` to skip it.


In [4]:
if RUN_PAPER_EVALUATION:
    paper_offline = BENCH.run_paper_evaluation("Offline")
    paper_offline_paired = BENCH.show_paired_analysis(
        *paper_offline,
        mode="Offline",
        methods=OFFLINE_METHODS,
        n_resamples=PAIRED_RESAMPLES
    )
    paper_online = BENCH.run_paper_evaluation("Streaming")
    paper_online_paired = BENCH.show_paired_analysis(
        *paper_online,
        mode="Streaming",
        methods=ONLINE_METHODS,
        n_resamples=PAIRED_RESAMPLES
    )
else:
    print("Expanded 449-track runs skipped (RUN_PAPER_EVALUATION=False).")


Reusing completed run: /Users/sarah/Desktop/shasha/Attune/benchmarks/results/pitch/notebook_runs/paper_offline_bach10-original/seed_0__per_instrument_None


,Tracks,Raw Pitch Accuracy,Raw Chroma Accuracy,Overall Accuracy,Voicing Recall,Voicing False Alarm,Audio(s)/Compute(s)
model,,,,,,,
pyin,40,0.9733,0.9742,0.9258,0.9844,0.4519,3.9530
attune,40,0.9722,0.9731,0.9382,0.9830,0.3317,7.3572
basic_pitch,40,0.9254,0.9263,0.8965,0.9692,0.3333,42.1262
praat,40,0.9620,0.9640,0.9363,0.9810,0.2680,275.3249
swiftf0,40,0.9426,0.9429,0.9041,0.9557,0.4021,42.0183
crepe,40,0.9742,0.9750,0.9330,0.9838,0.3944,0.1891
spice,40,0.8929,0.8940,0.8531,0.9367,0.4637,25.1765
rmvpe,40,0.9628,0.9640,0.9270,0.9823,0.3573,5.7627


No summary at /Users/sarah/Desktop/shasha/Attune/benchmarks/results/pitch/notebook_runs/paper_offline_urmp/seed_0__per_instrument_None/rows.csv; rebuilding scores from prediction caches.
  pyin: 149/149 cached predictions available for rescoring; only cache misses or incompatible configurations run detection
  attune: 149/149 cached predictions available for rescoring; only cache misses or incompatible configurations run detection
  basic_pitch: 121/149 cached predictions available for rescoring; only cache misses or incompatible configurations run detection
  praat: 26/149 cached predictions available for rescoring; only cache misses or incompatible configurations run detection
  swiftf0: 26/149 cached predictions available for rescoring; only cache misses or incompatible configurations run detection
  crepe: 26/149 cached predictions available for rescoring; only cache misses or incompatible configurations run detection
  spice: 26/149 cached predictions available for rescoring; only

KeyboardInterrupt: 

## Interpreting raw pitch accuracy

Raw pitch accuracy uses a 50-cent tolerance; raw chroma accuracy additionally ignores octave errors. Under these adapters' zero-frequency unvoiced convention, raw pitch accuracy decreases both when played tones are suppressed and when voiced estimates have the wrong pitch. The following decomposition uses pooled reference-voiced frame counts. It is not a mean of track-level percentages. Negative-frequency latent predictions, if introduced by another adapter, would require revisiting this decomposition.


In [ ]:
rpa_decomposition = BENCH.show_rpa_decomposition(
    preliminary, preliminary_urmp, preliminary_coco, methods=PRIMARY_METHODS
)


### Optional diagnostic: per-stratum breakdown

A pooled headline score can hide one instrument family carrying the loss. This splits the CocoChorales rows by (ensemble, instrument) so a method that is strong overall but weak on, say, the string ensemble is visible.

## Full comparisons

These cells drop the preliminary per-instrument sampler and run every track in the configured corpora, including all 40 original Bach10 isolated recordings and all 149 isolated real-recording URMP stems. They are long and cache-backed: jobs run in `CONFIG.workers` spawned workers with one numerical-library thread per worker, and any method whose dependency is missing is skipped rather than failing the run.

**Skipped by Run All by default.** Set `RUN_FULL_RUNS = True` in the configuration cell to enable these suites.
The full audio suite contains original Bach10 plus the two MDB synthesized corpora. Its new `all_audio_original_bach10` result directory preserves the previous `all_audio` results.


In [ ]:
if RUN_FULL_RUNS:
    all_audio = BENCH.run_all_audio()
else:
    print("Skipped full run (RUN_FULL_RUNS=False).")


In [ ]:
if RUN_FULL_RUNS:
    all_urmp = BENCH.run_all_urmp(force=False)
else:
    print("Skipped full run (RUN_FULL_RUNS=False).")


In [ ]:
if RUN_FULL_RUNS:
    all_coco = BENCH.run_all_coco(force=False)
else:
    print("Skipped full run (RUN_FULL_RUNS=False).")


## Leaderboard

One row per (method, suite), ranked by overall accuracy. Read `Voicing False Alarm` alongside it: the synthetic suites have effectively digital silence in their unvoiced frames, while original Bach10 and URMP supply real isolated recordings. Neither substitutes for a live microphone benchmark in a real room.

With `RUN_FULL_RUNS = False`, this table uses the three preliminary suites.

In [ ]:
leaderboard = BENCH.show_leaderboard(
    *(
        (all_audio, all_urmp, all_coco)
        if RUN_FULL_RUNS
        else (preliminary, preliminary_urmp, preliminary_coco)
    )
)


## Terminal equivalents

The same suites from a shell, for long overnight runs. `--write` publishes clean-suite frame-pooled metrics to `results/pitch/pitch_frame_weighted_benchmarks.csv`. Per-track results are written under `results/pitch/raw_outputs/<method>/`.

```shell
# balanced preliminary Bach10 draw, plus dependency availability
at-venv/bin/python benchmarks/modules/pitch/PitchBenchmarker.py --datasets bach10-original --per-instrument 2 --seed 0 --list

# full original Bach10 plus MDB synthesized corpora
at-venv/bin/python benchmarks/modules/pitch/PitchBenchmarker.py \
    --datasets mdb-stem-synth mdb-melody-synth bach10-original --write

# balanced preliminary URMP draw, then the complete 149-stem corpus
at-venv/bin/python benchmarks/modules/pitch/PitchBenchmarker.py --datasets urmp --per-instrument 2 --seed 0
at-venv/bin/python benchmarks/modules/pitch/PitchBenchmarker.py \
    --datasets urmp --write

# balanced preliminary CocoChorales draw
at-venv/bin/python benchmarks/modules/pitch/PitchBenchmarker.py \
    --datasets coco --split test --per-instrument 2 --seed 0 --materialize --write

# the heavy PyTorch models in their own invocation, so runtimes do not stack
at-venv/bin/python benchmarks/modules/pitch/PitchBenchmarker.py \
    --datasets bach10-original --methods torchcrepe penn --write

# resume an interrupted run, or re-score existing caches without detecting
at-venv/bin/python benchmarks/modules/pitch/PitchBenchmarker.py --datasets coco --skip-cached --write
at-venv/bin/python benchmarks/modules/pitch/PitchBenchmarker.py --datasets coco --cache-only --write
```